In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescent(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch * 100, max_patience=50)

epoch:  0, loss: 0.3824668824672699
epoch:  1, loss: 0.38062384724617004
epoch:  2, loss: 0.3787917494773865
epoch:  3, loss: 0.3769705593585968
epoch:  4, loss: 0.37516021728515625
epoch:  5, loss: 0.3733605444431305
epoch:  6, loss: 0.37157148122787476
epoch:  7, loss: 0.3697930574417114
epoch:  8, loss: 0.3680250346660614
epoch:  9, loss: 0.36626753211021423
epoch:  10, loss: 0.364520400762558
epoch:  11, loss: 0.36278364062309265
epoch:  12, loss: 0.3610570430755615
epoch:  13, loss: 0.3593405783176422
epoch:  14, loss: 0.3576342761516571
epoch:  15, loss: 0.35593798756599426
epoch:  16, loss: 0.35425159335136414
epoch:  17, loss: 0.35257506370544434
epoch:  18, loss: 0.3509083390235901
epoch:  19, loss: 0.34925132989883423
epoch:  20, loss: 0.347603976726532
epoch:  21, loss: 0.3459661900997162
epoch:  22, loss: 0.34433791041374207
epoch:  23, loss: 0.34271910786628723
epoch:  24, loss: 0.3411096930503845
epoch:  25, loss: 0.3395095765590668
epoch:  26, loss: 0.337918758392334
epo

KeyboardInterrupt: 

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.704505443572998
Test metrics:  R2 = 0.6771324276924133


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLipschitz(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.633337676525116
epoch:  1, loss: 0.04342888668179512
epoch:  2, loss: 0.03736550360918045
epoch:  3, loss: 0.037109244614839554
epoch:  4, loss: 0.03702034428715706
epoch:  5, loss: 0.03683463856577873
epoch:  6, loss: 0.036760617047548294
epoch:  7, loss: 0.0368603877723217
epoch:  8, loss: 0.036682385951280594
epoch:  9, loss: 0.036669958382844925
epoch:  10, loss: 0.036644916981458664
epoch:  11, loss: 0.03591656684875488
epoch:  12, loss: 0.04678220674395561
epoch:  13, loss: 0.04318079352378845
epoch:  14, loss: 0.035202283412218094
epoch:  15, loss: 0.0351484976708889
epoch:  16, loss: 0.03512383997440338
epoch:  17, loss: 0.03500339761376381
epoch:  18, loss: 0.03446457162499428
epoch:  19, loss: 0.03981679305434227
epoch:  20, loss: 0.03414740785956383
epoch:  21, loss: 0.0339910127222538
epoch:  22, loss: 0.0339510403573513
epoch:  23, loss: 0.033611930906772614
epoch:  24, loss: 0.03011414036154747
epoch:  25, loss: 0.16517478227615356
epoch:  26, loss: 0.0

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9589589834213257
Test metrics:  R2 = 0.9375925660133362


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model.parameters())

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=n_epoch, max_patience=50)

epoch:  0, loss: 0.33030128479003906
epoch:  1, loss: 0.18952080607414246
epoch:  2, loss: 0.11450280994176865
epoch:  3, loss: 0.07566405832767487
epoch:  4, loss: 0.05544142425060272
epoch:  5, loss: 0.045329850167036057
epoch:  6, loss: 0.040440306067466736
epoch:  7, loss: 0.038135919719934464
epoch:  8, loss: 0.03707012161612511
epoch:  9, loss: 0.03658339008688927
epoch:  10, loss: 0.036362648010253906
epoch:  11, loss: 0.036262549459934235
epoch:  12, loss: 0.03621666133403778
epoch:  13, loss: 0.036195024847984314
epoch:  14, loss: 0.036184124648571014
epoch:  15, loss: 0.03617795556783676
epoch:  16, loss: 0.03616628050804138
epoch:  17, loss: 0.03615123778581619
epoch:  18, loss: 0.03613455966114998
epoch:  19, loss: 0.03610619157552719
epoch:  20, loss: 0.03609145060181618
epoch:  21, loss: 0.036082860082387924
epoch:  22, loss: 0.036063302308321
epoch:  23, loss: 0.03604847565293312
epoch:  24, loss: 0.03604009002447128
epoch:  25, loss: 0.03602656349539757
epoch:  26, loss

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8120447397232056
Test metrics:  R2 = 0.779199481010437


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentTR(model.parameters(), radius_init=10)

model, loss_history = train_loop(model, loss_fn, opt, data_loader, epochs=1000, max_patience=50)

epoch:  0, loss: 0.6942681670188904
epoch:  1, loss: 0.6942681670188904
epoch:  2, loss: 0.05141329765319824
epoch:  3, loss: 0.05141329765319824
epoch:  4, loss: 0.040955815464258194
epoch:  5, loss: 0.040955815464258194
epoch:  6, loss: 0.036991480737924576
epoch:  7, loss: 0.036991480737924576
epoch:  8, loss: 0.03669799491763115
epoch:  9, loss: 0.03667027875781059
epoch:  10, loss: 0.03667027875781059
epoch:  11, loss: 0.03664816915988922
epoch:  12, loss: 0.03663856163620949
epoch:  13, loss: 0.036635518074035645
epoch:  14, loss: 0.03663266450166702
epoch:  15, loss: 0.03662988170981407
epoch:  16, loss: 0.0366271547973156
epoch:  17, loss: 0.036624472588300705
epoch:  18, loss: 0.036621835082769394
epoch:  19, loss: 0.036619242280721664
epoch:  20, loss: 0.03661669045686722
epoch:  21, loss: 0.036614175885915756
epoch:  22, loss: 0.036611709743738174
epoch:  23, loss: 0.03660928085446358
epoch:  24, loss: 0.03660690039396286
epoch:  25, loss: 0.03660455346107483
epoch:  26, los

In [ ]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -2962.131591796875
Test metrics:  R2 = -3712.926513671875
